In [1]:
def renumber_pdb(input_pdb, output_pdb, start=166):
    with open(input_pdb, 'r') as f:
        lines = f.readlines()
    
    current_resnum = None
    new_resnum = start - 1
    
    with open(output_pdb, 'w') as out:
        for line in lines:
            if line.startswith(('ATOM', 'HETATM')):
                res = line[22:26].strip()
                if res != current_resnum:
                    current_resnum = res
                    new_resnum += 1
                line = line[:22] + f'{new_resnum:4d}' + line[26:]
            out.write(line)


In [ ]:
input_file = "../1-loopmodel/target.BL00310001FH.pdb"

renumber_pdb(
        input_pdb = input_file,
        output_pdb = "../loop_model/PDE4B_modeller.pdb"
    )

In [8]:
import sys

def generate_cst_res(pdb_file, target_residues, anchor_res, output):
    coords = {}
    chains = {}

    with open(pdb_file) as f:
        for line in f:
            if line.startswith("ATOM") and line[12:16].strip() == "CA":
                res_num = int(line[22:26].strip())
                chain_id = line[21].strip()
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
                coords[res_num] = (x, y, z)
                chains[res_num] = chain_id

    if anchor_res not in coords:
        print(f"Erro: O resíduo âncora {anchor_res} não foi encontrado no PDB.")
        return

    anchor_chain = chains[anchor_res]
    constraints_count = 0
    with open(output, "w") as out:
        for res in target_residues:
            if res in coords:
                x, y, z = coords[res]
                res_chain = chains[res]
                out.write(
                    f"CoordinateConstraint CA {res}{res_chain} CA {anchor_res}{anchor_chain} "
                    f"{x:.3f} {y:.3f} {z:.3f} HARMONIC 0.0 1.0\n"
                )
                constraints_count += 1
            else:
                print(f"Aviso: Resíduo {res} especificado não foi encontrado no PDB e será pulado.")

    print(f"Gerado: {output} com {constraints_count} constraints criadas.")

In [9]:
target_residues = [405, 406, 519, 565, 567, 575, 578, 579, 582, 583, 586, 603, 614, 615, 618]

# Constrain catalytic site
generate_cst_res("../1-loopmodel\PDE4B_modeller.pdb", target_residues, anchor_res=405, output="cst2.cst")

Gerado: cst2.cst com 15 constraints criadas.


<>:4: SyntaxWarning: invalid escape sequence '\P'
<>:4: SyntaxWarning: invalid escape sequence '\P'
C:\Users\vitor\AppData\Local\Temp\ipykernel_10164\1131162462.py:4: SyntaxWarning: invalid escape sequence '\P'
  generate_cst_res("../1-loopmodel\PDE4B_modeller.pdb", target_residues, anchor_res=405, output="cst2.cst")


In [7]:
with open("../1-loopmodel\PDE4B_modeller.pdb") as f:
    for line in f:
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            res_num = int(line[22:26].strip())
            if res_num in (405, 406):
                print(repr(line[21]), res_num)

'A' 405
'A' 406


<>:1: SyntaxWarning: invalid escape sequence '\P'
<>:1: SyntaxWarning: invalid escape sequence '\P'
C:\Users\vitor\AppData\Local\Temp\ipykernel_10164\4039738931.py:1: SyntaxWarning: invalid escape sequence '\P'
  with open("../1-loopmodel\PDE4B_modeller.pdb") as f:
